In [133]:
import xarray as xr
import os
from os import listdir
from os.path import isfile, join
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

In [ ]:
def find_correct_year_file(read_directory,start_year,interval) :
    onlyfiles = [f for f in listdir(read_directory) if isfile(join(read_directory, f))]
    pattern = re.compile(f"{start_year}0101-{start_year+interval-1}123[0-1].nc$")
    for file in onlyfiles :
        match = pattern.search(file)
        if match is not None :
            break
    return file

In [128]:
def check_time_consistency(read_directory,interval=5,start_year=1951,end_year=2005) :
    """Checks that the time intervals are consistent with pre-defined parameters. 
    Returns True if files present in read_directory are consistent, False otherwise"""
    if (end_year-start_year+1)%interval != 0 :
        raise ValueError(f"Incorrect input. Inconsistency between start_year ({start_year}), end_year ({end_year}), and interval ({interval}).")
    onlyfiles = [f for f in listdir(read_directory) if isfile(join(read_directory, f))]
    for year in range(start_year,end_year,interval) :
        pattern = re.compile(f"{year}0101-{year+interval-1}123[0-1].nc$")
        flag = np.array([not(pattern.search(file) is None) for file in onlyfiles]).any() # flag is True if there is a match for one of the files, False otherwise
        if not flag :
            break
    return flag

In [129]:
check_time_consistency(os.getcwd(),5,2006,2100)

np.True_

In [ ]:
#os.chdir("../../Code")
os.getcwd()

['tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20810101-20851231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20360101-20401231.nc',
 '.git',
 'CORDEX.json',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20310101-20351231.nc',
 'CORDEX.ipynb',
 'export.csv',
 'catalogue.ipynb',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20860101-20901231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20610101-20651231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20460101-20501231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20760101-20801231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20560101-20601231.nc',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO2015_v1_day_20260101-20301231.nc',
 'CORDEX.csv.gz',
 'tas_EUR-11_CNRM-CERFACS-CNRM-CM5_rcp85_r1i1p1_GERICS-REMO

In [130]:
read_directory=os.getcwd()
write_directory=os.getcwd()
start_year_clim=2006
end_year_clim=2100
interval=5
temp_variable='tas'
smooth_span=15

In [134]:
onlyfiles = [f for f in listdir(read_directory) if isfile(join(read_directory, f))]
year_list = range(start_year_clim,end_year_clim,interval)

# Load files in a dictionary
for year in year_list :
    pattern = re.compile(f"{year}0101-{year+interval-1}123[0-1].nc$") #chose file from start year and interval
    ds_dict[year] = xr.open_dataset(join(read_directory,find_correct_year_file(read_directory,year,interval)), engine="netcdf4") 

# Initialize output data array with the first file
da = getattr(ds_dict[start_year_clim], temp_variable)

# Drop Feb 29
not_feb29 = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
da = da.sel(time=not_feb29)
# Create synthetic dayofyear: reset every year to 1–365
# Do this manually to break reliance on the original calendar
times = da.time
years = times.dt.year.values
months = times.dt.month.values
days = times.dt.day.values
# Create a structured datetime array to use as keys
# Create dummy dates all in a non-leap year (e.g., 2001)
uniform_dates = pd.to_datetime([f"2001-{month:02d}-{day:02d}" for month, day in zip(months, days)])
# Extract synthetic dayofyear 
dayofyear_365 = xr.DataArray(uniform_dates.dayofyear,coords={"time": times},dims="time")
# Group using this clean synthetic dayofyear and summing to compute mean at the end
climatology = da.groupby(dayofyear_365).sum("time")

for year in tqdm(year_list[1:]) : #iterate over files, except first one which has already been used in initialization
    da = getattr(ds_dict[start_year_clim], temp_variable)
    not_feb29 = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    da = da.sel(time=not_feb29)
    times = da.time
    years = times.dt.year.values
    months = times.dt.month.values
    days = times.dt.day.values
    uniform_dates = pd.to_datetime([f"2001-{month:02d}-{day:02d}" for month, day in zip(months, days)])
    dayofyear_365 = xr.DataArray(uniform_dates.dayofyear,coords={"time": times},dims="time")
    da = da.groupby(dayofyear_365).sum("time")
    climatology += da
    # Rename the dimension
climatology /= end_year_clim - start_year_clim +1
climatology = climatology.rename({"group": "dayofyear"})

100%|██████████| 18/18 [01:07<00:00,  3.74s/it]


In [144]:
extended_temp=np.zeros((365*3,np.shape(climatology.data)[1],np.shape(climatology.data)[2]))

In [ ]:
extended_temp[0:365,:,:]=climatology.data[:,:,:]
extended_temp[365:730,:,:]=climatology.data[:,:,:]
extended_temp[730:,:,:]=climatology.data[:,:,:]

In [160]:
# Smoothing
print("Smoothing...")
for i in tqdm(range(365,730)):
    val_table=np.array(np.zeros((2*smooth_span+1,np.shape(climatology.data)[1],np.shape(climatology.data)[2])))
    for j in range(-smooth_span,smooth_span+1,1):
        val_table[j]=extended_temp[i+j,:,:]
    climatology[i-365,:,:] = np.mean(val_table[:],axis=0)

Smoothing...


100%|██████████| 365/365 [00:05<00:00, 66.13it/s]


In [157]:
climatology.to_netcdf(os.path.join(write_directory,f"{temp_variable}_climatology_{start_year_clim}_{end_year_clim}.nc"))

In [161]:
climatology.close()